In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample2") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

# Устанавливаем таймауты и keep-alive как числа (без 's')
# Значения в секундах или миллисекундах (зависит от версии, обычно keepalivetime в сек)
hadoop_conf.set("fs.s3a.threads.keepalivetime", "60") 
hadoop_conf.set("fs.s3a.connection.timeout", "60000")
hadoop_conf.set("fs.s3a.attempts.maximum", "10")
hadoop_conf.set("fs.s3a.connection.establish.timeout", "5000")
hadoop_conf.set("fs.s3a.readahead.range", "65536")

hadoop_conf.set("fs.s3a.multipart.purge.age", "86400")

hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("DebeziumKafkaStream") \
    .config("spark.jars.packages", 
           "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/kafka-spark-checkpoint") \
    .getOrCreate()



In [ ]:
print(f"Spark version: {spark.version}") 

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType
from pyspark.sql.functions import from_json

kafka_bootstrap = "kafka:29092"
kafka_topic = "zakaz.avpalatov.zakaz_events"

# Схема Debezium JSON
schema = StructType([
    StructField("before", 
        StructType([
            StructField("id", LongType(), True),
            StructField("zakaz_id", IntegerType(), True),
            StructField("status", StringType(), True),
            StructField("ts", LongType(), True)
        ]), True),
    
    StructField("after", 
        StructType([
            StructField("id", LongType(), True),
            StructField("zakaz_id", IntegerType(), True),
            StructField("status", StringType(), True),
            StructField("ts", LongType(), True)  # микросекунды
        ]), True),
    
    StructField("source", 
        StructType([
            StructField("version", StringType(), True),
            StructField("connector", StringType(), True),
            StructField("name", StringType(), True),
            StructField("ts_ms", LongType(), True),
            StructField("snapshot", StringType(), True),
            StructField("db", StringType(), True),
            StructField("sequence", StringType(), True),
            StructField("schema", StringType(), True),
            StructField("table", StringType(), True),
            StructField("txId", LongType(), True),
            StructField("lsn", LongType(), True),
            StructField("xmin", StringType(), True)
        ]), True),
    
    StructField("op", StringType(), True),  # c=create, u=update, d=delete, r=read
    StructField("ts_ms", LongType(), True),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
    StructField("transaction", StringType(), True)
])


df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "earliest") \
    .load()

# Парсинг JSON
# parsed_df = df \
#     .selectExpr("CAST(value AS STRING) as json_str") \
#     .select(from_json("json_str", debezium_schema).alias("data")) \
#     .filter("data.after IS NOT NULL") \
#     .select("data.after.*", "data.op", "data.source.*")

# # Стриминг


# Распарсенные данные
json_df = df.selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json("json_str", schema).alias("data")) \
    .where("data.after IS NOT NULL") \
    .select("data.after.*")

# Вывод в консоль
json_df.writeStream \
    .format("console") \
    .option("truncate", False) \
    .start() \
    .awaitTermination()

26/02/04 14:18:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+----------------+
|189|2297    |new      |1770210780469650|
|190|9145    |in_proc  |1770210780469656|
|191|7614    |delayed  |1770210780469658|
|192|9478    |in_proc  |1770210780469660|
|189|2297    |delayed  |1770210782616591|
|177|2968    |delivered|1770210782616591|
|186|6532    |approved |1770210782616591|
|193|6357    |delayed  |1770210840890904|
|194|6663    |approved |1770210840890911|
|195|2897    |delivered|1770210840890913|
|196|1334    |approved |1770210840890915|
|190|9145    |approved |1770210842794237|
|155|5195    |delayed  |1770210842794237|
|160|2041    |approved |1770210842794237|
|175|6836    |approved |1770210842794237|
|185|2324    |delayed  |1770210842794237|
|197|4082    |cancelled|1770210900437075|
|198|4290    |approved |1770210900437080|
|199|6373    |new    

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/coder/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/coder/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

-------------------------------------------
Batch: 11
-------------------------------------------
-------------------------------------------
Batch: 7
-------------------------------------------
+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+----------------+
|465|6196    |cancelled|1770214920412304|
|466|4225    |new      |1770214920412312|
|467|2071    |delayed  |1770214920412313|
|468|3941    |delivered|1770214920412315|
+---+--------+---------+----------------+

+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+----------------+
|465|6196    |cancelled|1770214920412304|
|466|4225    |new      |1770214920412312|
|467|2071    |delayed  |1770214920412313|
|468|3941    |delivered|1770214920412315|
+---+--------+---------+----------------+

-------------------------------------------
Batch: 8
-------------------------------------------
----------------------------------

...

-------------------------------------------
Batch: 77
-------------------------------------------
Batch: 73
-------------------------------------------
-------------------------------------------
+---+--------+--------+----------------+
|id |zakaz_id|status  |ts              |
+---+--------+--------+----------------+
|582|2510    |approved|1770216782637108|
|591|5838    |in_proc |1770216782637108|
+---+--------+--------+----------------+

+---+--------+--------+----------------+
|id |zakaz_id|status  |ts              |
+---+--------+--------+----------------+
|582|2510    |approved|1770216782637108|
|591|5838    |in_proc |1770216782637108|
+---+--------+--------+----------------+

-------------------------------------------
Batch: 78
-------------------------------------------
-------------------------------------------
Batch: 74
-------------------------------------------
+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+------

-------------------------------------------
Batch: 97
-------------------------------------------
-------------------------------------------
Batch: 93
-------------------------------------------
+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+----------------+
|629|3078    |cancelled|1770217380415561|
|630|7153    |in_proc  |1770217380415568|
|631|3305    |delivered|1770217380415570|
|632|2060    |cancelled|1770217380415572|
+---+--------+---------+----------------+

+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+----------------+
|629|3078    |cancelled|1770217380415561|
|630|7153    |in_proc  |1770217380415568|
|631|3305    |delivered|1770217380415570|
|632|2060    |cancelled|1770217380415572|
+---+--------+---------+----------------+



-------------------------------------------
Batch: 98
-------------------------------------------
-------------------------------------------
Batch: 94
-------------------------------------------
+---+--------+--------+----------------+
|id |zakaz_id|status  |ts              |
+---+--------+--------+----------------+
|630|7153    |approved|1770217382414974|
+---+--------+--------+----------------+

+---+--------+--------+----------------+
|id |zakaz_id|status  |ts              |
+---+--------+--------+----------------+
|630|7153    |approved|1770217382414974|
+---+--------+--------+----------------+

-------------------------------------------
Batch: 99
-------------------------------------------
-------------------------------------------
Batch: 95
-------------------------------------------
+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+----------------+
|633|9403    |approved |1770217440759289|
|634|1345    |cancelled|1770

-------------------------------------------
Batch: 118
-------------------------------------------
-------------------------------------------
Batch: 114
-------------------------------------------
+---+--------+--------+----------------+
|id |zakaz_id|status  |ts              |
+---+--------+--------+----------------+
|669|8960    |approved|1770217982607701|
+---+--------+--------+----------------+

+---+--------+--------+----------------+
|id |zakaz_id|status  |ts              |
+---+--------+--------+----------------+
|669|8960    |approved|1770217982607701|
+---+--------+--------+----------------+

-------------------------------------------
Batch: 119
-------------------------------------------
-------------------------------------------
Batch: 115
-------------------------------------------
+---+--------+---------+----------------+
|id |zakaz_id|status   |ts              |
+---+--------+---------+----------------+
|673|8494    |approved |1770218040846628|
|674|5754    |new      |

-------------------------------------------
Batch: 212
-------------------------------------------
-------------------------------------------
Batch: 208
-------------------------------------------
+---+--------+-------+----------------+
|id |zakaz_id|status |ts              |
+---+--------+-------+----------------+
|853|6111    |new    |1770220740980585|
|854|9862    |new    |1770220740980590|
|855|2932    |new    |1770220740980592|
|856|3613    |in_proc|1770220740980594|
+---+--------+-------+----------------+

+---+--------+-------+----------------+
|id |zakaz_id|status |ts              |
+---+--------+-------+----------------+
|853|6111    |new    |1770220740980585|
|854|9862    |new    |1770220740980590|
|855|2932    |new    |1770220740980592|
|856|3613    |in_proc|1770220740980594|
+---+--------+-------+----------------+

-------------------------------------------
Batch: 213
-------------------------------------------
-------------------------------------------
Batch: 209
------

In [ ]:
spark.stop()